# YOLOv10 — pretrained inference on Dutch 17th-century interiors

Runs the pretrained YOLOv10 models (n, s, m, x variants) on your 84 annotated paintings, compares detections to your manual ground truth, and outputs:

- `yolo_predictions_<variant>.csv` — per-box detections
- `yolo_presence_metrics_<variant>.csv` — per-class precision/recall/F1
- `yolo_count_metrics_<variant>.csv` — per-class count MAE
- `yolo_confusion_<variant>.csv` — what GT objects each COCO prediction co-occurs with
- `overlays_<variant>/` — sample painting overlays (YOLO boxes + GT list)
- `yolo_model_comparison.csv` — headline numbers across all variants

## Before running

1. Upload your `rijks/` folder to Google Drive so it contains:
   - `Image Annotation - Annotations (updated).csv`
   - `Image Annotation - NEW Unique names list (updated).csv`
   - `analysis.py`
   - `yolo_eval.py`
   - `class_map.json`
   - `images/` — the 84 painting JPGs, filenames like `SK-A-2320.jpg` (the Picture ID with no suffix)
2. Runtime → Change runtime type → **T4 GPU**
3. Run cells top to bottom.


## 1. Install dependencies

In [ ]:
!pip install -q ultralytics pandas pillow
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ↓↓↓ EDIT THIS to point at your rijks folder on Drive ↓↓↓
RIJKS = '/content/drive/MyDrive/rijks'

IMAGES_DIR   = f'{RIJKS}/images'
ANNOT_CSV    = f'{RIJKS}/Image Annotation - Annotations (updated).csv'
UNIQUE_CSV   = f'{RIJKS}/Image Annotation - NEW Unique names list (updated).csv'
CLASS_MAP    = f'{RIJKS}/class_map.json'
OUT_DIR      = f'{RIJKS}/yolo_outputs'

import os, sys
sys.path.append(RIJKS)   # so we can `import analysis, yolo_eval`
os.makedirs(OUT_DIR, exist_ok=True)

import glob
pics = sorted(glob.glob(f'{IMAGES_DIR}/*.jpg') + glob.glob(f'{IMAGES_DIR}/*.jpeg'))
print(f'{len(pics)} painting images found under {IMAGES_DIR}')

## 3. Load ground truth + class map

In [ ]:
import pandas as pd
from analysis import load_and_normalize
from yolo_eval import (load_class_map, aggregate_predictions,
                       aggregate_ground_truth, presence_metrics,
                       count_metrics, confusion_matrix, summary_row,
                       draw_overlay)

ann, uniq = load_and_normalize(ann_path=ANNOT_CSV, uniq_path=UNIQUE_CSV)
coco_to_gt, gt_to_coco = load_class_map(CLASS_MAP)
print(f'Ground-truth annotations: {len(ann):,} rows across {ann["pic_base"].nunique()} paintings')
print(f'Mapped COCO classes: {len(coco_to_gt)}')
gt_long = aggregate_ground_truth(ann, gt_to_coco)
print('GT rolled up into', len(gt_long), 'painting × coco-class rows')
gt_long.head()

## 4. Run YOLOv10 on every painting

We test four model sizes to match your existing YOLOv8 n/s/m comparison on slide 43. Ultralytics auto-downloads the weights on first use.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import re

VARIANTS = ['yolov10n', 'yolov10s', 'yolov10m', 'yolov10x']
CONF_THRESHOLD = 0.25   # bump up to 0.4-0.5 if you want fewer FPs

def pic_base_from_path(p: str) -> str:
    stem = Path(p).stem.upper().replace(' ', '')
    return re.sub(r'-[ABC]$', '', stem)

all_results = {}
for variant in VARIANTS:
    print(f'\n=== {variant} ===')
    model = YOLO(f'{variant}.pt')
    rows = []
    for batch_start in range(0, len(pics), 16):
        batch = pics[batch_start:batch_start + 16]
        results = model(batch, conf=CONF_THRESHOLD, verbose=False)
        for path, res in zip(batch, results):
            pic = pic_base_from_path(path)
            if res.boxes is None:
                continue
            for b in res.boxes:
                cls_id = int(b.cls.item())
                cls_name = model.names[cls_id]
                conf = float(b.conf.item())
                x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
                rows.append({
                    'pic_base': pic, 'image_path': path,
                    'coco_class': cls_name, 'confidence': conf,
                    'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                })
    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(f'{OUT_DIR}/yolo_predictions_{variant}.csv', index=False)
    all_results[variant] = pred_df
    print(f'  {len(pred_df):,} boxes across {pred_df["pic_base"].nunique()} paintings')

## 5. Compute evaluation metrics for each variant

In [ ]:
paintings = list(ann['pic_base'].unique())
summary_rows = []

for variant, pred_df in all_results.items():
    pred_long = aggregate_predictions(pred_df, coco_to_gt)
    presence = presence_metrics(pred_long, gt_long, paintings)
    counts = count_metrics(pred_long, gt_long)
    conf_m = confusion_matrix(pred_df, ann, coco_to_gt)

    presence.to_csv(f'{OUT_DIR}/yolo_presence_metrics_{variant}.csv', index=False)
    counts.to_csv(f'{OUT_DIR}/yolo_count_metrics_{variant}.csv', index=False)
    conf_m.to_csv(f'{OUT_DIR}/yolo_confusion_{variant}.csv', index=False)

    row = {'model': variant, 'detections': len(pred_df),
           **summary_row(presence, counts)}
    summary_rows.append(row)
    print(f'{variant}: macro P={row["macro_precision"]}  R={row["macro_recall"]}  '
          f'F1={row["macro_F1"]}  count-MAE={row["mean_count_MAE"]}')

comparison = pd.DataFrame(summary_rows)
comparison.to_csv(f'{OUT_DIR}/yolo_model_comparison.csv', index=False)
comparison

## 6. Headline per-class performance (using YOLOv10x, the biggest)

Shows where YOLO out-of-the-box does well (person, dog, chair) vs. where it fails (domain-specific items it was never trained on).

In [ ]:
best = 'yolov10x'
presence = pd.read_csv(f'{OUT_DIR}/yolo_presence_metrics_{best}.csv')
counts   = pd.read_csv(f'{OUT_DIR}/yolo_count_metrics_{best}.csv')

print('\n=== Presence (ranked by F1) ===')
print(presence.sort_values('F1', ascending=False).to_string(index=False))
print('\n=== Count MAE (biggest gaps first) ===')
print(counts.to_string(index=False))

## 7. Generate sample overlays

In [ ]:
import os
os.makedirs(f'{OUT_DIR}/overlays_{best}', exist_ok=True)

pred_df = all_results[best]
SAMPLE_PICS = paintings[:10]   # first 10 paintings; change as you like

for pic in SAMPLE_PICS:
    pred_rows = pred_df[pred_df['pic_base'] == pic]
    if pred_rows.empty:
        print(f'{pic}: no YOLO detections, skipping')
        continue
    image_path = pred_rows.iloc[0]['image_path']
    gt_rows = ann[ann['pic_base'] == pic]
    gt_list = []
    for _, r in gt_rows.iterrows():
        gt_list.extend([r['obj_canonical']] * int(r['cnt']))
    out = f'{OUT_DIR}/overlays_{best}/{pic}.png'
    draw_overlay(image_path, pred_rows, gt_list, out)
    print(f'  {out}')

## 8. Bar chart: macro-F1 across YOLO variants

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(comparison['model'], comparison['macro_F1'], color='#2b6cb0')
axes[0].set_title('Macro F1 by YOLOv10 variant')
axes[0].set_ylabel('F1')
axes[0].set_ylim(0, 1)
for i, v in enumerate(comparison['macro_F1']):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center')

axes[1].bar(comparison['model'], comparison['mean_count_MAE'], color='#ed8936')
axes[1].set_title('Mean Count MAE by variant (lower is better)')
axes[1].set_ylabel('MAE')
for i, v in enumerate(comparison['mean_count_MAE']):
    axes[1].text(i, v + 0.05, f'{v:.2f}', ha='center')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/yolo_variants_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

## Done.

Check `yolo_outputs/` in your Drive — the CSVs there are the table-ready numbers for your slide deck. The overlays PNGs are in `overlays_yolov10x/` to drop straight onto slides.

**Next step:** open `yolov10_finetune.ipynb` once you have the Grounding DINO bbox output.